In [2]:
from openai import OpenAI
import openai
import os

api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = api_key

In [1]:
%pip install wikipedia

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11703 sha256=4f0a555cc817ee799468cbf410b89bf85a90dc5b38facc7c52524053f8ce20c6
  Stored in directory: c:\users\knri0\appdata\local\pip\cache\wheels\b2\7f\26\524faff9145e274da278dc97d63ab0bfde1f791ecf101a9c95
Successfully built wikipedia
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import wikipedia

In [9]:
wikipedia.set_lang('ko')
result = wikipedia.search('세종대왕')[0]
wikipedia.summary(result, sentences=5)

'세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 훈민정음은 언문으로 불리며 왕실과 민간에서 사용되다가 20세기 주시경이 한글로 발전시켜 오늘날 대한민국의 공식 문자로서 널리 쓰이고 있다.'

In [6]:
def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}

In [7]:
wikipedia_search("세종대왕은 누구야?" )

{'summary': '세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 훈민정음은 언문으로 불리며 왕실과 민간에서 사용되다가 20세기 주시경이 한글로 발전시켜 오늘날 대한민국의 공식 문자로서 널리 쓰이고 있다.'}

In [8]:
import json
import wikipedia
from openai import OpenAI


client = OpenAI()


def wikipedia_search(question):
    wikipedia.set_lang("ko")
    try:
        search_result = wikipedia.search(question)[0]
        wiki_summary = wikipedia.summary(search_result, sentences=5)
    except Exception as e:
        wiki_summary = "위키피디아에서 정보를 찾을 수 없습니다."
    return {"summary": wiki_summary}


wikifunc = [{
    "type": "function",
    "function": {
        "name": "wikipedia_search",
        "description": "입력된 질문에 대해 필요하다면 위키피디아에서  정보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": " 주제 또는 질문"
                }
            },
            "required": ["question"]
        }
    }
}]

messages = [{"role": "user", "content": "세종대왕에 대해 알려줘"}]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    tools=wikifunc,
    tool_choice="auto"
)


tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
print( args)
result = wikipedia_search(args["question"])
print(result)

# messages = [{"role": "user", "content": "오리 너구리에 대해 알려줄래"},
#             {"role": "function","name": "wikipedia_search",
#              "content":'{summany:위키피디아답변}'}]
messages.append({
    "role": "function",
    "name": "wikipedia_search",
    "content": json.dumps(result)
})

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0
)


print(response.choices[0].message.content)


{'question': '세종대왕'}
{'summary': '세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 훈민정음은 언문으로 불리며 왕실과 민간에서 사용되다가 20세기 주시경이 한글로 발전시켜 오늘날 대한민국의 공식 문자로서 널리 쓰이고 있다.'}
세종대왕(한국어: 世宗, 중세 한국어: ·世宗, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위: 1418년 9월 9일~1450년 3월 30일)으로, 도한과 현종의 아버지이다. 현대인들은 세종의 업적을 평가하며 세종의 지혜를 높이 평가하고 있다.

세종대왕은 학문, 과학, 문학, 국방 등 다양한 분야에서 혁신적인 업적을 이루었으며, 백성들에게 유익한 정책을 실시하여 조선시대의 번영을 이끌었다. 세종대왕은 국민들에게 교육을 제공하고 문화를 발전시키며, 한글을 창제하여 국민들의 의사소통을 촉진시켰다. 또한 20세기 초에는 한글을 복원하여 현대 한국어의 기초를 다지고 발전시킨 것으로 평가받고 있다.
